# CourtListener search → DataFrame (HCDE 530) — **`week5_courtlistener_cases.ipynb`**

Uses the **Week 4** folder and reads **`COURTLISTENER_API_TOKEN`** from **`week 4/.env`** (same variable as `week4_public_api.py`). Supports `KEY=value` or `export KEY=value` in `.env`.

Install packages into the **same** interpreter as this notebook (global kernel): `python3 -m pip install -r "week 4/requirements.txt"` — or use the Week 5 venv if you use one.

Calls CourtListener **Legal Search API v4**: `GET /api/rest/v4/search/` with `type=o` (opinion clusters).

The API returns **`caseName`**, **`judge`**, and **`dateFiled`**. Plaintiff and defendant are **not** separate API fields; we **parse** them from `caseName` when it looks like *Party A v. Party B* (otherwise those columns are missing).

A second table (**Judgments classified by Judge names and jurisdiction**) pulls **20** opinions from the Seattle-area federal court (`court_id:wawd`) with **`jurisdiction`** set to **`Seattle jurisdiction`** on every row.

A third section (**Most recent judgments**) loads King County–related decisions from the **Washington Court of Appeals** (`court_id:washctapp` + phrase `"King County"`), sorts by decision date, and shows the **20 newest** rows (see notes there on coverage).

**Part 2** stacks those pulls for **pandas** and runs the five in-class table operations (`head` / `info`, `value_counts`, boolean filter, `groupby` + `mean`, `isnull` counts) on the combined CourtListener dataset.


## Part 2 — Pandas analysis (MP1-style)

The cells below treat the three CourtListener tables as **one assignment dataset**: combined where helpful, with plain-English `#` comments tied to each **class operation**.

**Five in-class operations → this notebook**

| Operation | Question it answers | Where / column |
|-----------|---------------------|----------------|
| `df.head()` and `df.info()` | What does the data look like? dtypes and non-null counts? | Stacked `df_mp1` |
| `df['column'].value_counts()` | What judge strings appear most in the Seattle federal sample? | `df_jurisdiction['judge']` |
| `df[df['column'] >= value]` | Filter to decisions on or after a cutoff (here: 1990-01-01) | `df_mp1['_decision_dt']` vs `pd.Timestamp('1990-01-01')` |
| `df.groupby('column')['other'].mean()` | Mean `cite_count` by dataset slice after filtering | `df_recent_window.groupby('dataset')['cite_count']` |
| `df.isnull().sum()` | Missing cells per column across the stacked frame | `df_mp1` |


In [5]:
from __future__ import annotations

from IPython.display import display

import json
import os
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

# Find folder that contains `week 4/.env` or `.env` (kernel cwd varies)
HERE = Path.cwd()
for candidate in (HERE, HERE / "week 4", HERE.parent / "week 4"):
    if (candidate / ".env").is_file():
        HERE = candidate
        break
ENV_PATH = HERE / ".env"
TOKEN_ENV = "COURTLISTENER_API_TOKEN"
SEARCH_URL = "https://www.courtlistener.com/api/rest/v4/search/"


def load_dotenv_file(path: Path) -> None:
    if not path.is_file():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        if key.lower().startswith("export "):
            key = key[7:].strip()
        val = val.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = val


def split_parties(case_name: str) -> tuple[str, str]:
    if not case_name or not isinstance(case_name, str):
        return "", ""
    for sep in (" v. ", " V. ", " vs. ", " VS. ", " v ", " V "):
        if sep in case_name:
            left, right = case_name.split(sep, 1)
            return left.strip(), right.strip()
    return "", ""


def courtlistener_search(query: str, *, opinion_type: str = "o", token: str | None) -> dict:
    params = {"q": query, "type": opinion_type}
    url = SEARCH_URL + "?" + urllib.parse.urlencode(params)
    headers = {"Accept": "application/json; indent=2"}
    if token:
        headers["Authorization"] = f"Token {token}"
    req = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        detail = e.read().decode("utf-8", errors="replace")[:800]
        raise RuntimeError(f"CourtListener HTTP {e.code}: {detail}") from e


load_dotenv_file(ENV_PATH)
token = os.environ.get(TOKEN_ENV)
if not token:
    raise RuntimeError(
        f"Missing {TOKEN_ENV}. Add to {ENV_PATH} (see .env.example). Do not commit .env."
    )

QUERY = "Miranda v. Arizona"
raw = courtlistener_search(QUERY, token=token)
results = raw.get("results") or []
print("total matches:", raw.get("count"))
print("rows on first page:", len(results))


total matches: 47040
rows on first page: 20


In [6]:
rows: list[dict[str, object]] = []
for item in results[:25]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    plaintiff, defendant = split_parties(case)
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    cite = item.get("citeCount")
    try:
        cite_n = int(cite) if cite is not None else 0
    except (TypeError, ValueError):
        cite_n = 0

    rows.append(
        {
            "case": case,
            "plaintiff": plaintiff or pd.NA,
            "defendant": defendant or pd.NA,
            "judge": judge or pd.NA,
            "date_of_decision": date_dec or pd.NA,
            "cite_count": cite_n,
        }
    )

df = pd.DataFrame(rows)
df


,case,plaintiff,defendant,judge,date_of_decision,cite_count
0,Miranda v. Arizona,Miranda,Arizona,"Consideration, Took",1969-10-13,46
1,Miranda v. Arizona,Miranda,Arizona,NaN,1965-11-22,3
2,Miranda v. Arizona,Miranda,Arizona,"Warren, Clark, Stewart, White, Harlan",1966-06-13,58080
3,Best v. Miranda,Best,Miranda,"Brown, Swann, Thompson",2012-03-15,1
4,State v. Miranda-Cabrera,State,Miranda-Cabrera,"Snow, Timmer, Ehrlich",2004-12-02,42
5,State v. Miranda,State,Miranda,"Charles, Feldman, Jones, McGREGOR, Stanley, Th...",2001-05-04,7
6,"Callan, Miranda, Azuelo... v. Pimber","Callan, Miranda, Azuelo...",Pimber,NaN,2006-03-22,0
7,State v. Miranda,State,Miranda,"Timmer, Toci, Gerber",2000-09-28,18
8,State of Arizona v. Richard J. Glassel,State of Arizona,Richard J. Glassel,"Bales, Berch, Pelander",2013-11-21,0
9,People v. Miranda-Guerrero,People,Miranda-Guerrero,NaN,2022-11-17,31


## Judgments classified by Judge names and jurisdiction

Federal **Seattle** area opinions use CourtListener’s **`court_id:wawd`** (U.S. District Court, **Western District of Washington**). Each row sets **`jurisdiction`** to the label **`Seattle jurisdiction`**; **`court`** is the API’s full court name.

## Most recent judgments (King County)

CourtListener does **not** give King County **Superior** Court its own `court_id`. To capture **King County** decisions that are published in the corpus, this query uses the **Court of Appeals of Washington** (`court_id:washctapp`) plus the phrase **`"King County"`** (appeals from King County Superior Court and other King County matters often appear here).

The API returns pages in relevance order, so the code **follows `next`**, collects results, **sorts by `dateFiled` descending**, and keeps the **top 20** most recent opinions.

In [7]:
from datetime import date


def _opinion_date(item: dict) -> date:
    ds = item.get("dateFiled") or ""
    parts = ds.split("-")
    if len(parts) != 3:
        return date.min
    y, m, d = (int(parts[0]), int(parts[1]), int(parts[2]))
    return date(y, m, d)


def collect_search_pages(q: str, *, token: str, max_pages: int = 25) -> list[dict]:
    """GET /search with pagination via `next` URL."""
    headers = {"Accept": "application/json; indent=2", "Authorization": f"Token {token}"}
    url = SEARCH_URL + "?" + urllib.parse.urlencode({"q": q, "type": "o"})
    acc: list[dict] = []
    for _ in range(max_pages):
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=60) as resp:
            payload = json.loads(resp.read().decode("utf-8"))
        acc.extend(payload.get("results") or [])
        url = payload.get("next")
        if not url or len(acc) >= 1200:
            break
    return acc


QUERY_KING_COUNTY = '"King County" court_id:washctapp'
raw_kc_count = courtlistener_search(QUERY_KING_COUNTY, token=token)
print("King County–indexed matches (Wash. Ct. App.):", raw_kc_count.get("count"))

king_items = collect_search_pages(QUERY_KING_COUNTY, token=token)
king_items.sort(key=_opinion_date, reverse=True)
king_top20 = king_items[:20]
print("rows paginated for sorting:", len(king_items), "· using newest", len(king_top20), "by dateFiled")

rows_recent: list[dict[str, object]] = []
for item in king_top20:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    cite = item.get("citeCount")
    try:
        cite_n = int(cite) if cite is not None else 0
    except (TypeError, ValueError):
        cite_n = 0

    rows_recent.append(
        {
            "case": case or pd.NA,
            "judge": judge or pd.NA,
            "court": (item.get("court") or "").strip() or pd.NA,
            "court_id": (item.get("court_id") or "").strip() or pd.NA,
            "date_filed": item.get("dateFiled") or pd.NA,
            "cite_count": cite_n,
        }
    )

df_recent = pd.DataFrame(rows_recent)
df_recent

King County–indexed matches (Wash. Ct. App.): 21254
rows paginated for sorting: 500 · using newest 20 by dateFiled


,case,judge,court,court_id,date_filed,cite_count
0,King County V. Aquatherm Gmbh,NaN,Court of Appeals of Washington,washctapp,2026-03-23,0
1,In Re The Dependency Of H.w.,NaN,Court of Appeals of Washington,washctapp,2025-07-21,0
2,"Fall City Sustainable Growth, V. King County",NaN,Court of Appeals of Washington,washctapp,2025-05-19,0
3,"Russell Carter, V. Multicare Health System",NaN,Court of Appeals of Washington,washctapp,2024-07-30,0
4,"Grady Austin, V. King County",NaN,Court of Appeals of Washington,washctapp,2024-07-02,0
5,"Skycorp, Ltd., V. King County",NaN,Court of Appeals of Washington,washctapp,2024-02-13,0
6,"King County, V. Walsh Construction Company Ii,...",NaN,Court of Appeals of Washington,washctapp,2023-07-03,0
7,"King County, V. Friends Of Sammamish Valley",NaN,Court of Appeals of Washington,washctapp,2023-06-12,0
8,"City Of Sammamish, V. John Titcomb, Jr., Linde...",NaN,Court of Appeals of Washington,washctapp,2023-03-13,2
9,Jared Karstetter Et Ano. V. King County Correc...,NaN,Court of Appeals of Washington,washctapp,2022-08-29,0


In [8]:
SEATTLE_JURISDICTION_LABEL = "Seattle jurisdiction"

raw_seattle = courtlistener_search("court_id:wawd", token=token)
results_seattle = raw_seattle.get("results") or []
print("Seattle-area (W.D. Wash.) total matches:", raw_seattle.get("count"))
print("rows on this page (using first 20):", min(20, len(results_seattle)))

rows_jurisdiction: list[dict[str, object]] = []
for item in results_seattle[:20]:
    case = (item.get("caseName") or item.get("caseNameFull") or "").strip()
    judge = (item.get("judge") or "").strip()
    panel = item.get("panel_names") or []
    if not judge and panel:
        judge = "; ".join(str(p) for p in panel if p)
    court_name = (item.get("court") or "").strip()
    court_id = (item.get("court_id") or "").strip()
    date_dec = item.get("dateFiled") or item.get("dateArgued")

    cite = item.get("citeCount")
    try:
        cite_n = int(cite) if cite is not None else 0
    except (TypeError, ValueError):
        cite_n = 0

    rows_jurisdiction.append(
        {
            "judge": judge or pd.NA,
            "jurisdiction": SEATTLE_JURISDICTION_LABEL,
            "court": court_name or pd.NA,
            "court_id": court_id or pd.NA,
            "case": case or pd.NA,
            "date_of_decision": date_dec or pd.NA,
            "cite_count": cite_n,
        }
    )

df_jurisdiction = pd.DataFrame(rows_jurisdiction)
df_jurisdiction

Seattle-area (W.D. Wash.) total matches: 3037
rows on this page (using first 20): 20


,judge,jurisdiction,court,court_id,case,date_of_decision,cite_count
0,NaN,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Simmons v. Safeway, Inc.",2019-08-01,0
1,Settle,Seattle jurisdiction,"District Court, W.D. Washington",wawd,State v. Franciscan Health Sys.,2019-03-01,1
2,Jones,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Beane v. RPW Legal Servs., PLLC",2019-05-06,2
3,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Animal Legal Def. Fund v. Olympic Game Farm, Inc.",2019-05-21,3
4,Lasnik,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Galvez v. Cuccinelli,2019-07-17,4
5,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Mitchell v. Atkins,2019-05-20,1
6,Robart,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Calderon-Rodriguez v. Wilcox,2019-02-06,1
7,Pechman,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Gallupe v. Sedgwick Claims Mgmt. Servs. Inc.,2019-02-14,3
8,Leighton,Seattle jurisdiction,"District Court, W.D. Washington",wawd,Marine Carpenters Pension Fund v. Puglia Marin...,2019-04-10,0
9,Martinez,Seattle jurisdiction,"District Court, W.D. Washington",wawd,"Pengbo Xiao v. Feast Buffet, Inc.",2019-05-15,5


### Question 1 — What does the combined CourtListener dataset look like?

We stack the three pulls into **`df_mp1`** so one frame carries Miranda search rows, W.D. Wash. rows, and King County appellate rows. **`head()`** shows example values; **`info()`** shows dtypes and how many non-null values exist per column.

In [ ]:
    # Question 1: After stacking all three API slices, what do the first rows look like and what dtypes / non-null counts do we have?
# Answer meaning: head() shows real rows so we can sanity-check names and dates; info() tells us which columns are sparse or object-typed before we filter or group.

df_m = df.assign(dataset="Miranda search")
df_j = df_jurisdiction.assign(dataset="W.D. Wash. (Seattle)")
df_r = df_recent.rename(columns={"date_filed": "date_of_decision"}).assign(
    dataset="King County (Wash. Ct. App.)"
)
df_mp1 = pd.concat([df_m, df_j, df_r], ignore_index=True, sort=False)

display(df_mp1.head(10))
df_mp1.info()

,case,plaintiff,defendant,judge,date_of_decision,cite_count,dataset,jurisdiction,court,court_id
0,Miranda v. Arizona,Miranda,Arizona,"Consideration, Took",1969-10-13,46,Miranda search,NaN,NaN,NaN
1,Miranda v. Arizona,Miranda,Arizona,NaN,1965-11-22,3,Miranda search,NaN,NaN,NaN
2,Miranda v. Arizona,Miranda,Arizona,"Warren, Clark, Stewart, White, Harlan",1966-06-13,58080,Miranda search,NaN,NaN,NaN
3,Best v. Miranda,Best,Miranda,"Brown, Swann, Thompson",2012-03-15,1,Miranda search,NaN,NaN,NaN
4,State v. Miranda-Cabrera,State,Miranda-Cabrera,"Snow, Timmer, Ehrlich",2004-12-02,42,Miranda search,NaN,NaN,NaN
5,State v. Miranda,State,Miranda,"Charles, Feldman, Jones, McGREGOR, Stanley, Th...",2001-05-04,7,Miranda search,NaN,NaN,NaN
6,"Callan, Miranda, Azuelo... v. Pimber","Callan, Miranda, Azuelo...",Pimber,NaN,2006-03-22,0,Miranda search,NaN,NaN,NaN
7,State v. Miranda,State,Miranda,"Timmer, Toci, Gerber",2000-09-28,18,Miranda search,NaN,NaN,NaN
8,State of Arizona v. Richard J. Glassel,State of Arizona,Richard J. Glassel,"Bales, Berch, Pelander",2013-11-21,0,Miranda search,NaN,NaN,NaN
9,People v. Miranda-Guerrero,People,Miranda-Guerrero,NaN,2022-11-17,31,Miranda search,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   case              60 non-null     str  
 1   plaintiff         20 non-null     str  
 2   defendant         20 non-null     str  
 3   judge             33 non-null     str  
 4   date_of_decision  60 non-null     str  
 5   cite_count        60 non-null     int64
 6   dataset           60 non-null     str  
 7   jurisdiction      20 non-null     str  
 8   court             40 non-null     str  
 9   court_id          40 non-null     str  
dtypes: int64(1), str(9)
memory usage: 4.8 KB


### Question 2 — Which judge labels are most common in the Seattle federal sample?

The W.D. Wash. table keeps one **judge** string per opinion (sometimes empty). **`value_counts()`** ranks those labels so we see concentration versus variety on this page of results.

In [10]:
# Question 2: In the Seattle-area federal pull, which judge field strings show up most often on this page?
# Answer meaning: value_counts() tells us whether a few last names dominate the relevance-ranked first page or whether the mix is flat — it describes the *sample*, not who is “most important” statewide.

df_jurisdiction["judge"].value_counts(dropna=False)

judge
Settle                    3
Leighton                  3
Robart                    3
Lasnik                    2
Pechman                   2
Martinez                  2
NaN                       1
Jones                     1
Introduction, Leighton    1
Theiler                   1
Zilly                     1
Name: count, dtype: int64

### Question 3 — How sparse is the data, and how do post-1990 citation averages compare by slice?

**`isnull().sum()`** counts missing cells per column. A **boolean filter** keeps decisions on or after 1990 so older Miranda noise does not dominate. **`groupby(...).mean()`** on **`cite_count`** compares average indexed citations across the three pulls.

In [11]:
# Question 3a: Across all stacked rows, how many missing values does each column have?
# Answer meaning: isnull().sum() is a completeness audit — large counts (e.g., judge) warn us that person-based charts would be biased or empty for many rows.
missing_per_col = df_mp1.isnull().sum()
display(missing_per_col)

# Question 3b: Which opinions have a decision date on or after 1990-01-01?
# Answer meaning: filtering to a recent window keeps the citation comparison from being dragged down by decades-old Miranda hits that legitimately have different citation patterns.
df_mp1["_decision_dt"] = pd.to_datetime(df_mp1["date_of_decision"], errors="coerce")
df_recent_window = df_mp1[df_mp1["_decision_dt"] >= pd.Timestamp("1990-01-01")]
print("rows with decision date >= 1990-01-01:", len(df_recent_window))
display(df_recent_window.head(6))

# Question 3c: Inside that window, what is the mean cite_count for each dataset label?
# Answer meaning: groupby(dataset)['cite_count'].mean() summarizes CourtListener's citeCount field — low averages mean few later cases cite these opinions in the index, not that the underlying law is unimportant.
df_recent_window.groupby("dataset", dropna=False)["cite_count"].mean().round(2)

case                 0
plaintiff           40
defendant           40
judge               27
date_of_decision     0
cite_count           0
dataset              0
jurisdiction        40
court               20
court_id            20
dtype: int64

rows with decision date >= 1990-01-01: 57


,case,plaintiff,defendant,judge,date_of_decision,cite_count,dataset,jurisdiction,court,court_id,_decision_dt
3,Best v. Miranda,Best,Miranda,"Brown, Swann, Thompson",2012-03-15,1,Miranda search,NaN,NaN,NaN,2012-03-15
4,State v. Miranda-Cabrera,State,Miranda-Cabrera,"Snow, Timmer, Ehrlich",2004-12-02,42,Miranda search,NaN,NaN,NaN,2004-12-02
5,State v. Miranda,State,Miranda,"Charles, Feldman, Jones, McGREGOR, Stanley, Th...",2001-05-04,7,Miranda search,NaN,NaN,NaN,2001-05-04
6,"Callan, Miranda, Azuelo... v. Pimber","Callan, Miranda, Azuelo...",Pimber,NaN,2006-03-22,0,Miranda search,NaN,NaN,NaN,2006-03-22
7,State v. Miranda,State,Miranda,"Timmer, Toci, Gerber",2000-09-28,18,Miranda search,NaN,NaN,NaN,2000-09-28
8,State of Arizona v. Richard J. Glassel,State of Arizona,Richard J. Glassel,"Bales, Berch, Pelander",2013-11-21,0,Miranda search,NaN,NaN,NaN,2013-11-21


dataset
King County (Wash. Ct. App.)     1.90
Miranda search                  21.35
W.D. Wash. (Seattle)             3.40
Name: cite_count, dtype: float64

### Notes

- **Pagination**: follow the `next` URL (cursor) for more pages; cache is ~10 minutes per CourtListener docs.
- **King County**: **Most recent judgments** uses `court_id:washctapp` + phrase `"King County"` because King County Superior Court is not its own CourtListener `court_id`; appellate opinions tied to King County supply recent dates.
- **Token**: same `COURTLISTENER_API_TOKEN` as `week4_public_api.py` in `week 4/.env`.
- **Docs**: [Legal Search API](https://www.courtlistener.com/help/api/rest/search/)
